# سبق 07 - پلاننگ ڈیزائن پیٹرن

یہ نوٹ بک مائیکروسافٹ ایجنٹ فریم ورک استعمال کرتے ہوئے AI ایجنٹس کے لئے **پلاننگ ڈیزائن پیٹرن** کو ظاہر کرتی ہے۔
آپ سیکھیں گے کہ پیچیدہ سفر کی درخواستوں کو منظم ذیلی کاموں میں کیسے تقسیم کیا جائے، انہیں ماہر ایجنٹس کو تفویض کیا جائے،
اور نتیجہ خیز منصوبہ کو انجام دیا جائے — سب کچھ اسٹرکچرد آؤٹ پٹ کے ذریعے جو Pydantic ماڈلز کے ذریعے چلایا گیا ہے۔


## ترتیب


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## کام کی تقسیم

کام کی تقسیم منصوبہ بندی کے ڈیزائن پیٹرن کا مرکز ہے۔ بجائے اس کے کہ ایک ہی ایجنٹ سے کہا جائے کہ وہ
پیچیدہ درخواست کو ابتدا سے انتہا تک سنبھالے، ہم مسئلہ کو چھوٹے، واضح **ذیلی کاموں** میں تقسیم کرتے ہیں۔
ہر ذیلی کام کو ایک ماہر ایجنٹ (مثلاً، پروازیں، ہوٹل، سرگرمیاں) کو تفویض کیا جاتا ہے جس کی واضح
ترجیحات اور انحصار کی ترتیب ہوتی ہے۔

اس طریقہ کار کے کئی فوائد ہیں:
- **وضاحت**: ہر ذیلی کام کی ایک مخصوص ذمہ داری ہوتی ہے۔
- **متوازی عمل**: آزاد ذیلی کام بیک وقت چل سکتے ہیں۔
- **قابلِ اعتماد**: ناکامیاں انفرادی ذیلی کاموں تک محدود رہتی ہیں۔
- **بجٹ کی نگرانی**: اخراجات ہر ذیلی کام کے لحاظ سے تخمینہ کیے جاتے ہیں اور مجموعی طور پر جمع ہوتے ہیں۔


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ساختہ آؤٹ پٹ کے ساتھ ایک منصوبہ بندی ایجنٹ بنانا

منصوبہ بندی کا ایجنٹ ایک **فرنٹ ڈیسک کوآرڈینیٹر** کے طور پر کام کرتا ہے۔ ایک اعلیٰ سطح کے سفر کی درخواست دی جانے پر یہ
ایک ساختہ `TravelPlan` تیار کرتا ہے — درخواست کو ذیلی کاموں میں تقسیم کرنا، ترجیحات مقرر کرنا،
اور انحصار کی شناخت کرنا تاکہ کوئی کانسیئر یا نفاذ کرنے والا طبقہ کام انجام دے سکے۔


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ماہر آلات کے ساتھ منصوبہ بندی کرنا  

جب فرنٹ ڈیسک ایجنٹ ایک منظم منصوبہ تیار کر لیتا ہے، تو **کونسیئرج ایجنٹ** اسے عملی جامہ پہنتا ہے۔  
ہر ماہر آلہ ذیلی کام کی ایک قسم (پروازیں، ہوٹل، سرگرمیاں) کو سنبھالتا ہے۔ کونسیئرج  
منصوبے کے ذیلی کاموں کو ان کے انحصار کی ترتیب میں دہراتا ہے اور ہر ایک کو متعلقہ آلے کو بھیجتا ہے۔  
 


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## خلاصہ

اس سبق میں آپ نے AI ایجنٹوں کے لیے **پلاننگ ڈیزائن پیٹرن** سیکھا:

1. **ٹاسک تجزیہ** — ایک فرنٹ ڈیسک پلاننگ ایجنٹ پیچیدہ سفر کی درخواست کو
   Pydantic ماڈلز کے استعمال سے منظم ذیلی کاموں میں توڑتا ہے، ہر ایک کو ایک ماہر ایجنٹ کے ساتھ ترجیحات
   اور انحصارات تفویض کرتا ہے۔
2. **منظم پیداوار** — `response_format` پاس کرکے ایجنٹ آزاد قسم کے متن کی بجائے ایک مستند
   `TravelPlan` آبجیکٹ واپس کرتا ہے، جس سے نچلے عمل کی قابلِ اعتماد پروسیسنگ ممکن ہوتی ہے۔
3. **منصوبہ بندی کا نفاذ** — ایک کنسیرج ایجنٹ ماہر اوزاروں
   (`book_flight`, `reserve_hotel`, `book_activity`) کا استعمال کرتے ہوئے ذیلی کاموں کو انجام دیتا ہے اور نتائج رپورٹ کرتا ہے۔

یہ پیٹرن *کیا کرنا ہے* (پلاننگ) کو *کیسے کرنا ہے* (نفاذ) سے الگ کرتا ہے، جس سے ایجنٹس
زیادہ ماڈیولر، قابلِ جانچ اور آسانی سے بڑھائے جانے والے بنتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
